# Comparison experiment: evaluate the released checkpoints

Loads the seven compact baseline checkpoints in `checkpoints/` (implemented in `train_baseline.py`) and evaluates them on the test set (62,002 windows) without training.
The reported numbers were recorded during GPU training (`released_results/summary_baselines_seed163.csv`).

In [1]:
import os, sys, importlib.util
os.environ.setdefault("DATA_DIR", "../data/features_21d")
import tempfile
os.environ["OUT_DIR"] = tempfile.mkdtemp()   # train_baseline.py creates OUT_DIR on import; nothing is written to it here
_spec = importlib.util.spec_from_file_location("train_baseline", "train_baseline.py")
TB = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(TB)   # definitions + data only; no training
import torch
from sklearn.metrics import f1_score
import pandas as pd

ref = pd.read_csv("released_results/summary_baselines_seed163.csv").set_index("model")
rows = []
for n in TB.RUN_NAMES:
    m = TB.BUILDERS[n]().to(TB.DEVICE)
    m.load_state_dict(torch.load("checkpoints/best_" + n.replace("-", "") + ".pth", map_location="cpu"))
    _, acc, probs = TB.evaluate(m, TB.te_loader)
    pred = probs.argmax(1)
    rows.append({"model": n, "params": TB.n_params(m), "test_correct": int((pred == TB.y_te).sum()),
                 "test_acc_%": round(100 * acc, 4), "macro_f1": round(f1_score(TB.y_te, pred, average="macro"), 4),
                 "released_correct": int(ref.loc[n, "test_correct"])})
pd.DataFrame(rows)

device = cuda | epochs = 100 | batch = 2048 | seed = 163
runs   = ['TabNet', 'FT-Transformer', 'AutoInt', 'DCN-V2', 'ExcelFormer', 'Tab-ResNet', 'NODE']
features = 21 | classes = ['Aging', 'Fault', 'Noise', 'Normal']
train/val/test = (197001, 21) (21000, 21) (62002, 21)
train label counts = [49251 49250 49250 49250]
model                  params
TabNet                   1076
FT-Transformer           4044
AutoInt                  4658
DCN-V2                   3997
ExcelFormer             16100
Tab-ResNet              18948
NODE                     2404


,model,params,test_correct,test_acc_%,macro_f1,released_correct
0,TabNet,1076,60813,98.0823,0.9808,60813
1,FT-Transformer,4044,61230,98.7549,0.9875,61230
2,AutoInt,4658,61269,98.8178,0.9882,61269
3,DCN-V2,3997,61040,98.4484,0.9844,61040
4,ExcelFormer,16100,61010,98.4001,0.9840,61010
5,Tab-ResNet,18948,60731,97.9501,0.9794,60731
6,NODE,2404,61317,98.8952,0.9889,61317
